# Prueba tecnica ENKI

## Paso 1: Extracción de datos con Python

### 1.1 Consumo de la API

In [1]:
import requests

url_openmeteo = "https://api.open-meteo.com/v1/forecast"

end_point = "?latitude=19.43&longitude=-99.13&hourly=temperature_2m,precipitation&past_days=7&forecast_days=0"

response = requests.get(url=url_openmeteo + end_point)

if response.status_code == 200:
    data = response.json()
    print("Solicitud exitosa. Datos obtenidos correctamente.")
    
else:
    raise Exception(f"Error en la API: {response.status_code}")

Solicitud exitosa. Datos obtenidos correctamente.


In [2]:
print(data)

{'latitude': 19.437609, 'longitude': -99.10715, 'generationtime_ms': 0.05042552947998047, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 2238.0, 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm'}, 'hourly': {'time': ['2026-04-20T00:00', '2026-04-20T01:00', '2026-04-20T02:00', '2026-04-20T03:00', '2026-04-20T04:00', '2026-04-20T05:00', '2026-04-20T06:00', '2026-04-20T07:00', '2026-04-20T08:00', '2026-04-20T09:00', '2026-04-20T10:00', '2026-04-20T11:00', '2026-04-20T12:00', '2026-04-20T13:00', '2026-04-20T14:00', '2026-04-20T15:00', '2026-04-20T16:00', '2026-04-20T17:00', '2026-04-20T18:00', '2026-04-20T19:00', '2026-04-20T20:00', '2026-04-20T21:00', '2026-04-20T22:00', '2026-04-20T23:00', '2026-04-21T00:00', '2026-04-21T01:00', '2026-04-21T02:00', '2026-04-21T03:00', '2026-04-21T04:00', '2026-04-21T05:00', '2026-04-21T06:00', '2026-04-21T07:00', '2026-04-21T08:00', '2026-04-21T09:00', '2026-04-21T10:00', '2026-04-

In [3]:
hourly = data.get("hourly", {})

time = hourly.get("time", [])
temperature = hourly.get("temperature_2m", [])
precipitation = hourly.get("precipitation", [])

In [4]:
records = []

for t, temp, prec in zip(time, temperature, precipitation):
    records.append({
        "time": t,
        "temperature_2m": temp,
        "precipitation": prec
    })

In [5]:
records

[{'time': '2026-04-20T00:00', 'temperature_2m': 21.4, 'precipitation': 0.0},
 {'time': '2026-04-20T01:00', 'temperature_2m': 19.1, 'precipitation': 1.3},
 {'time': '2026-04-20T02:00', 'temperature_2m': 19.9, 'precipitation': 0.0},
 {'time': '2026-04-20T03:00', 'temperature_2m': 19.8, 'precipitation': 0.0},
 {'time': '2026-04-20T04:00', 'temperature_2m': 17.2, 'precipitation': 0.0},
 {'time': '2026-04-20T05:00', 'temperature_2m': 16.1, 'precipitation': 0.0},
 {'time': '2026-04-20T06:00', 'temperature_2m': 15.2, 'precipitation': 0.0},
 {'time': '2026-04-20T07:00', 'temperature_2m': 14.5, 'precipitation': 0.0},
 {'time': '2026-04-20T08:00', 'temperature_2m': 14.6, 'precipitation': 0.0},
 {'time': '2026-04-20T09:00', 'temperature_2m': 14.4, 'precipitation': 0.0},
 {'time': '2026-04-20T10:00', 'temperature_2m': 14.1, 'precipitation': 0.0},
 {'time': '2026-04-20T11:00', 'temperature_2m': 13.6, 'precipitation': 0.0},
 {'time': '2026-04-20T12:00', 'temperature_2m': 13.6, 'precipitation': 0.0},

### 1.2 Limpieza y transformación de datos

In [6]:
import pandas as pd 

df = pd.DataFrame(records)
df['time'] = pd.to_datetime(df["time"])
df.columns = ["fecha", "temperatura_c", "precipitacion_mm"]
df.head()

,fecha,temperatura_c,precipitacion_mm
0,2026-04-20 00:00:00,21.4,0.0
1,2026-04-20 01:00:00,19.1,1.3
2,2026-04-20 02:00:00,19.9,0.0
3,2026-04-20 03:00:00,19.8,0.0
4,2026-04-20 04:00:00,17.2,0.0


In [7]:
df = df[df["fecha"].dt.hour.between(6,22)]

In [8]:
df.set_index("fecha", inplace=True)

In [9]:
# valores nulos 
df.isnull().sum()

temperatura_c       0
precipitacion_mm    0
dtype: int64

In [10]:
df[(df["temperatura_c"] < 0) | (df["precipitacion_mm"] < 0) ] 

,temperatura_c,precipitacion_mm
fecha,,


In [11]:
df.to_csv("datos_clima_cdmx.csv")

## Paso 2: Análisis con SQL

In [ ]:
import sqlite3
import csv

conn = sqlite3.connect("base_ENKI.db")
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS clima_cdmx (
               fecha TIMESTAP,
               temperatura_c NUMERIC,
               precipitacion_mm NUMERIC,
               PRIMARY KEY (fecha)
               ) """)

with open("datos_clima_cdmx.csv", newline='', encoding='utf-8') as archivo:
    reader = csv.reader(archivo)
    
    next(reader)  # saltar encabezado
    
    for fila in reader:
        cursor.execute(
            "INSERT INTO clima_cdmx (fecha, temperatura_c, precipitacion_mm) VALUES (?, ?, ?)", 
            fila
        )

conn.commit()
conn.close()

In [14]:
def run_query(query, db="base_ENKI.db"):
        
    conn = sqlite3.connect(db)
    cursor = conn.cursor()
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    for row in rows:
        print(row)

    conn.close()

### Consulta A — Temperatura promedio por día

In [15]:
query1 = """SELECT 
                DATE(fecha) AS fecha,
                AVG(temperatura_c) AS avg_temperatura_c
            FROM clima_cdmx
            GROUP BY 1
            ORDER BY 2 DESC
        """

run_query(query1)


('2026-04-26', 21.15294117647059)
('2026-04-25', 20.494117647058825)
('2026-04-22', 19.42941176470588)
('2026-04-24', 19.323529411764707)
('2026-04-21', 18.835294117647063)
('2026-04-23', 18.04705882352941)
('2026-04-20', 17.66470588235294)


### Consulta B — Horas con precipitación

In [16]:
query2 = """SELECT 
                fecha,
                precipitacion_mm
            FROM clima_cdmx
            WHERE precipitacion_mm > 0 
            ORDER BY 1 
        """

run_query(query2)

('2026-04-20 18:00:00', 0.1)
('2026-04-21 10:00:00', 0.2)
('2026-04-21 11:00:00', 0.2)
('2026-04-22 22:00:00', 0.1)
('2026-04-23 22:00:00', 0.1)
('2026-04-24 22:00:00', 1.6)
('2026-04-26 22:00:00', 0.4)


### Consulta C — Día con mayor variación térmica

In [17]:
query3 = """
         SELECT 
            DATE(fecha),
            MAX(temperatura_c) AS temperatura_max,
            MIN(temperatura_c) AS temperatura_min,
            MAX(temperatura_c) - MIN(temperatura_c) AS variacion_termica
         FROM clima_cdmx
         GROUP BY 1
         ORDER BY 4 DESC
         LIMIT 1
            
         """

run_query(query3)

('2026-04-24', 28.7, 12.6, 16.1)


### Consulta D — Resumen diario

In [18]:
query4 = """
            SELECT 
                DATE(fecha) AS dia,
                MIN(temperatura_c) AS min_temperatura_c,
                MAX(temperatura_c) AS max_temperatura_c,
                AVG(temperatura_c) AS avg_temperatura_c,
                SUM(precipitacion_mm) AS precipitacion_acumulada_mm
            FROM clima_cdmx
            GROUP BY 1
            ORDER BY 1
        """

run_query(query4)

('2026-04-20', 13.5, 24.3, 17.66470588235294, 0.1)
('2026-04-21', 13.8, 25.9, 18.835294117647063, 0.4)
('2026-04-22', 15, 25.1, 19.42941176470588, 0.1)
('2026-04-23', 12.2, 25.5, 18.04705882352941, 0.1)
('2026-04-24', 12.6, 28.7, 19.323529411764707, 1.6)
('2026-04-25', 14.2, 28.8, 20.494117647058825, 0)
('2026-04-26', 15.6, 28.6, 21.15294117647059, 0.4)


## Paso 3: Documentación Técnica

1. Flujo del pipeline.

El pipeline realiza la extracción de datos climáticos desde la API de Open-Meteo utilizando requests. 
Posteriormente, los datos son transformados con pandas: se convierten a formato datetime, se renombran las columnas y se filtran los registros entre las 06:00 y las 22:00 horas. 
Se aplican validaciones básicas para identificar valores nulos y negativos en las variables numéricas. 
Finalmente, los datos limpios se exportan a un archivo CSV y se cargan en una base de datos  para su posterior consulta.

2. Decisiones Tecnicas.

Se utilizó la librería pandas para la transformación de datos debido a su eficiencia en el manejo de estructuras tabulares y operaciones de limpieza. 
Para el consumo de la API se empleó requests, ya que es una solución simple y ampliamente utilizada para llamadas HTTP en Python. 
Se implementó manejo de errores con try/except para controlar fallos en la conexión o respuestas inválidas de la API. 

3. Mejoras futuras 

Si tuviera más tiempo, implementaría un manejo más robusto de errores y validaciones, incluyendo alertas automáticas en caso de fallos. 
También integraría el pipeline en un scheduler (como cron o Airflow) para su ejecución periódica. 
Finalmente, consideraría almacenar los datos en una base de datos más escalable (como PostgreSQL) y agregar pruebas unitarias para garantizar la calidad del código.